In [ ]:
import os
import json
import subprocess
import webbrowser
from datetime import datetime
from urllib.parse import quote

from dotenv import load_dotenv
from openai import OpenAI


# ============================================================
# KENZIE v0.3
# Assistente pessoal para Windows 10
#
# Recursos:
# - Memória permanente
# - Modo offline
# - IA online opcional
# - Controle básico do Windows
# - Abertura de programas
# - Abertura de sites
# - Pesquisa na internet
# - Comandos naturais
# ============================================================


# ============================================================
# CONFIGURAÇÃO
# ============================================================

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY")

client = None

if API_KEY:

    try:
        client = OpenAI(api_key=API_KEY)

    except Exception:
        client = None


MEMORY_FILE = "kenzie_memory.json"

MAX_MESSAGES = 20

conversation = []


# ============================================================
# PERSONALIDADE
# ============================================================

SYSTEM_PROMPT = """
Você é Kenzie, uma assistente pessoal de inteligência artificial.

Personalidade:

- Inteligente
- Educada
- Natural
- Confiante
- Prestativa
- Objetiva
- Calma
- Humor sutil quando apropriado

Você está funcionando em um computador Windows 10.

Regras:

1. Responda em português quando o usuário falar português.
2. Seja natural.
3. Não invente informações.
4. Não diga que executou uma ação que não executou.
5. Não execute ações perigosas sem confirmação.
6. Utilize informações da memória quando forem relevantes.
7. Seja objetiva.
8. Avise quando alguma capacidade ainda não estiver implementada.
"""


# ============================================================
# MEMÓRIA
# ============================================================

def load_memory():

    if not os.path.exists(MEMORY_FILE):
        return {}

    try:

        with open(
            MEMORY_FILE,
            "r",
            encoding="utf-8"
        ) as file:

            return json.load(file)

    except Exception:

        return {}


def save_memory(memory):

    try:

        with open(
            MEMORY_FILE,
            "w",
            encoding="utf-8"
        ) as file:

            json.dump(
                memory,
                file,
                ensure_ascii=False,
                indent=4
            )

        return True

    except Exception:

        return False


memory = load_memory()


def show_memory():

    if not memory:

        return "Minha memória está vazia."

    result = "Estas são as informações que lembro:\n\n"

    for key, value in memory.items():

        result += f"- {key}: {value}\n"

    return result


# ============================================================
# CONTEXTO
# ============================================================

def add_user_message(message):

    conversation.append({
        "role": "user",
        "content": message
    })

    limit_context()


def add_assistant_message(message):

    conversation.append({
        "role": "assistant",
        "content": message
    })

    limit_context()


def limit_context():

    if len(conversation) > MAX_MESSAGES:

        del conversation[:-MAX_MESSAGES]


def clear_context():

    conversation.clear()


# ============================================================
# INFORMAÇÕES DO SISTEMA
# ============================================================

def get_time():

    return datetime.now().strftime("%H:%M:%S")


def get_date():

    return datetime.now().strftime("%d/%m/%Y")


# ============================================================
# ABRIR PROGRAMAS
# ============================================================

def open_calculator():

    try:

        subprocess.Popen("calc.exe")

        return "Calculadora aberta."

    except Exception as error:

        return f"Não consegui abrir a calculadora: {error}"


def open_notepad():

    try:

        subprocess.Popen("notepad.exe")

        return "Bloco de Notas aberto."

    except Exception as error:

        return f"Não consegui abrir o Bloco de Notas: {error}"


def open_paint():

    try:

        subprocess.Popen("mspaint.exe")

        return "Paint aberto."

    except Exception as error:

        return f"Não consegui abrir o Paint: {error}"


def open_explorer():

    try:

        subprocess.Popen("explorer.exe")

        return "Explorador de Arquivos aberto."

    except Exception as error:

        return f"Não consegui abrir o Explorador: {error}"


def open_task_manager():

    try:

        subprocess.Popen("taskmgr.exe")

        return "Gerenciador de Tarefas aberto."

    except Exception as error:

        return f"Não consegui abrir o Gerenciador de Tarefas: {error}"


# ============================================================
# SITES
# ============================================================

def open_google():

    webbrowser.open("https://www.google.com")

    return "Google aberto."


def open_youtube():

    webbrowser.open("https://www.youtube.com")

    return "YouTube aberto."


def open_github():

    webbrowser.open("https://github.com")

    return "GitHub aberto."


def search_google(query):

    url = (
        "https://www.google.com/search?q="
        + quote(query)
    )

    webbrowser.open(url)

    return f"Pesquisando por: {query}"


# ============================================================
# DOCUMENTOS
# ============================================================

def list_documents():

    documents = os.path.join(
        os.path.expanduser("~"),
        "Documents"
    )

    if not os.path.exists(documents):

        return "A pasta Documentos não foi encontrada."

    try:

        files = os.listdir(documents)

        if not files:

            return "A pasta Documentos está vazia."

        result = "Arquivos encontrados:\n\n"

        for file in files[:50]:

            result += f"- {file}\n"

        return result

    except Exception as error:

        return f"Não consegui acessar Documentos: {error}"


# ============================================================
# SISTEMA
# ============================================================

def lock_computer():

    try:

        subprocess.run(
            ["rundll32.exe", "user32.dll,LockWorkStation"],
            check=False
        )

        return "Computador bloqueado."

    except Exception as error:

        return f"Não consegui bloquear o computador: {error}"


# ============================================================
# MEMÓRIA
# ============================================================

def process_memory_command(command):

    text = command.strip()

    lower = text.lower()


    # --------------------------------------------------------
    # NOME
    # --------------------------------------------------------

    prefixes = [
        "meu nome é ",
        "meu nome e "
    ]

    for prefix in prefixes:

        if lower.startswith(prefix):

            value = text[len(prefix):].strip()

            if value:

                memory["nome"] = value

                save_memory(memory)

                return (
                    f"Prazer, {value}. "
                    "Vou guardar seu nome."
                )


    # --------------------------------------------------------
    # LEMBRAR
    # --------------------------------------------------------

    if lower.startswith("lembre que "):

        value = text[len("lembre que "):].strip()

        if value:

            key = f"informação_{len(memory) + 1}"

            memory[key] = value

            save_memory(memory)

            return (
                "Certo. Guardei essa informação "
                "na minha memória."
            )


    # --------------------------------------------------------
    # MOSTRAR MEMÓRIA
    # --------------------------------------------------------

    if (
        "o que você lembra" in lower
        or "o que voce lembra" in lower
        or "mostre sua memória" in lower
        or "mostre sua memoria" in lower
    ):

        return show_memory()


    # --------------------------------------------------------
    # NOME
    # --------------------------------------------------------

    if (
        "qual é meu nome" in lower
        or "qual e meu nome" in lower
        or "você sabe meu nome" in lower
        or "voce sabe meu nome" in lower
    ):

        if "nome" in memory:

            return f"Seu nome é {memory['nome']}."

        return "Ainda não sei seu nome."


    return None


# ============================================================
# COMANDOS LOCAIS
# ============================================================

def process_local_command(command):

    # Remove espaços extras
    # e transforma tudo em minúsculas.
    text = command.strip().lower()

    # Remove "kenzie" do começo da frase.
    # Assim:
    #
    # "Kenzie, abra o paint"
    #
    # vira:
    #
    # "abra o paint"

    if text.startswith("kenzie"):

        text = text[6:].strip()

        if text.startswith(","):
            text = text[1:].strip()


    # ========================================================
    # SAIR
    # ========================================================

    if text in [
        "sair",
        "exit",
        "quit",
        "encerrar",
        "fechar"
    ]:

        return "__EXIT__"


    # ========================================================
    # LIMPAR CONVERSA
    # ========================================================

    if any(
        phrase in text
        for phrase in [
            "limpar conversa",
            "limpe a conversa",
            "apagar conversa",
            "resetar conversa"
        ]
    ):

        return "__CLEAR__"


    # ========================================================
    # HORA
    # ========================================================

    if (
        "que horas" in text
        or "qual a hora" in text
        or text == "hora"
    ):

        return f"Agora são {get_time()}."


    # ========================================================
    # DATA
    # ========================================================

    if (
        "qual a data" in text
        or "qual é a data" in text
        or "que dia é hoje" in text
        or "que dia e hoje" in text
        or text == "data"
    ):

        return f"Hoje é {get_date()}."


    # ========================================================
    # CALCULADORA
    # ========================================================

    if "calculadora" in text:

        if any(
            word in text
            for word in [
                "abra",
                "abrir",
                "abre",
                "inicie",
                "iniciar",
                "execute",
                "executar",
                "rode",
                "rodar"
            ]
        ):

            return open_calculator()


    # ========================================================
    # PAINT
    # ========================================================

    if "paint" in text:

        if any(
            word in text
            for word in [
                "abra",
                "abrir",
                "abre",
                "inicie",
                "iniciar",
                "execute",
                "executar",
                "rode",
                "rodar"
            ]
        ):

            return open_paint()


    # ========================================================
    # BLOCO DE NOTAS
    # ========================================================

    if (
        "bloco de notas" in text
        or "notepad" in text
    ):

        if any(
            word in text
            for word in [
                "abra",
                "abrir",
                "abre",
                "inicie",
                "iniciar",
                "execute",
                "executar",
                "rode",
                "rodar"
            ]
        ):

            return open_notepad()


    # ========================================================
    # EXPLORADOR DE ARQUIVOS
    # ========================================================

    if (
        "explorador" in text
        or "explorer" in text
        or "arquivos" in text
    ):

        if any(
            word in text
            for word in [
                "abra",
                "abrir",
                "abre",
                "inicie",
                "iniciar",
                "execute",
                "executar",
                "rode",
                "rodar"
            ]
        ):

            return open_explorer()


    # ========================================================
    # GERENCIADOR DE TAREFAS
    # ========================================================

    if (
        "gerenciador de tarefas" in text
        or "task manager" in text
    ):

        if any(
            word in text
            for word in [
                "abra",
                "abrir",
                "abre",
                "inicie",
                "iniciar",
                "execute",
                "executar",
                "rode",
                "rodar"
            ]
        ):

            return open_task_manager()


    # ========================================================
    # GOOGLE
    # ========================================================

    if "google" in text:

        if any(
            word in text
            for word in [
                "abra",
                "abrir",
                "abre",
                "acesse",
                "acessar"
            ]
        ):

            return open_google()


    # ========================================================
    # YOUTUBE
    # ========================================================

    if "youtube" in text:

        if any(
            word in text
            for word in [
                "abra",
                "abrir",
                "abre",
                "acesse",
                "acessar"
            ]
        ):

            return open_youtube()


    # ========================================================
    # GITHUB
    # ========================================================

    if "github" in text:

        if any(
            word in text
            for word in [
                "abra",
                "abrir",
                "abre",
                "acesse",
                "acessar"
            ]
        ):

            return open_github()


    # ========================================================
    # PESQUISA NA INTERNET
    # ========================================================

    search_prefixes = [
        "pesquise ",
        "pesquisa ",
        "procure ",
        "buscar ",
        "busque ",
        "pesquisa no google ",
        "pesquise no google "
    ]


    for prefix in search_prefixes:

        if text.startswith(prefix):

            query = text[len(prefix):].strip()

            if query:

                return search_google(query)


    # ========================================================
    # DOCUMENTOS
    # ========================================================

    if (
        "liste meus documentos" in text
        or "listar documentos" in text
        or "mostre meus documentos" in text
        or "ver meus documentos" in text
        or "mostrar documentos" in text
    ):

        return list_documents()


    # ========================================================
    # BLOQUEAR COMPUTADOR
    # ========================================================

    if (
        "bloqueie o computador" in text
        or "bloquear computador" in text
        or "bloqueia o computador" in text
        or "trave o computador" in text
        or "travar computador" in text
    ):

        return lock_computer()


    # ========================================================
    # NENHUM COMANDO ENCONTRADO
    # ========================================================

    return None
# ============================================================
# MODO OFFLINE
# ============================================================

def offline_response(message):

    text = message.lower().strip()


    if text in [
        "oi",
        "olá",
        "ola",
        "oi kenzie",
        "olá kenzie",
        "ola kenzie"
    ]:

        return (
            "Olá! Eu sou a Kenzie. "
            "Estou funcionando em modo offline."
        )


    if (
        "quem é você" in text
        or "quem e voce" in text
    ):

        return (
            "Eu sou a Kenzie, sua assistente pessoal."
        )


    if (
        "como você está" in text
        or "como voce esta" in text
    ):

        return "Estou funcionando normalmente."


    if (
        "obrigado" in text
        or "obrigada" in text
    ):

        return "Por nada."


    if "ajuda" in text:

        return (
            "Posso controlar algumas funções do Windows, "
            "abrir programas e sites, pesquisar na internet, "
            "consultar hora e data e usar minha memória."
        )


    return (
        "Estou funcionando offline no momento. "
        "Ainda não tenho um modelo de IA disponível "
        "para responder perguntas complexas."
    )


# ============================================================
# IA
# ============================================================

def ask_ai(message):

    if client is None:

        return offline_response(message)


    add_user_message(message)


    memory_text = ""

    if memory:

        memory_text = (
            "\n\nInformações lembradas:\n"
        )

        for key, value in memory.items():

            memory_text += (
                f"- {key}: {value}\n"
            )


    messages = [

        {
            "role": "system",
            "content": (
                SYSTEM_PROMPT
                + memory_text
            )
        }

    ]


    messages.extend(conversation)


    try:

        response = client.chat.completions.create(

            model="gpt-4o-mini",

            messages=messages,

            temperature=0.7

        )


        answer = response.choices[0].message.content


        add_assistant_message(answer)


        return answer


    except Exception as error:

        error_text = str(error)


        if "429" in error_text:

            return (
                "A IA online está sem créditos. "
                "Continuarei funcionando offline."
            )


        if "401" in error_text:

            return (
                "A chave da API não foi aceita. "
                "Verifique o arquivo .env."
            )


        return (
            "Não consegui acessar a IA agora. "
            "Posso continuar usando meus recursos offline."
        )


# ============================================================
# BANNER
# ============================================================

def show_banner():

    print()

    print("=" * 60)

    print("                     KENZIE v0.3")

    print("              ASSISTENTE PESSOAL DE IA")

    print("=" * 60)

    print()

    print("Sistema: Windows 10")

    print("Memória: ATIVA")

    if client:

        print("IA online: CONFIGURADA")

    else:

        print("IA online: INDISPONÍVEL")

    print()

    print("Kenzie online.")
    print("Como posso ajudar?")

    print()

    print("Exemplos:")

    print("  Kenzie, abra a calculadora")
    print("  Kenzie, abra o Paint")
    print("  Kenzie, abra o Google")
    print("  Kenzie, pesquise inteligência artificial")
    print("  Kenzie, abra o Explorador")
    print("  Kenzie, bloqueie o computador")
    print("  Kenzie, o que você lembra?")
    print()

    print("Digite 'sair' para encerrar.")

    print()


# ============================================================
# MAIN
# ============================================================

def main():

    show_banner()


    while True:

        try:

            user_input = input("Você: ").strip()


            if not user_input:

                continue


            # MEMÓRIA

            memory_result = process_memory_command(
                user_input
            )


            if memory_result is not None:

                print()
                print(f"Kenzie: {memory_result}")
                print()

                continue


            # COMANDOS LOCAIS

            local_result = process_local_command(
                user_input
            )


            if local_result == "__EXIT__":

                print()
                print("Kenzie: Até logo.")
                print()

                break


            if local_result == "__CLEAR__":

                clear_context()

                print()
                print("Kenzie: Contexto limpo.")
                print()

                continue


            if local_result is not None:

                print()
                print(f"Kenzie: {local_result}")
                print()

                continue


            # IA

            print()

            print("Kenzie: Pensando...")

            answer = ask_ai(
                user_input
            )

            print()

            print(f"Kenzie: {answer}")

            print()


        except KeyboardInterrupt:

            print()
            print("Kenzie: Até logo.")
            print()

            break


        except Exception as error:

            print()
            print(
                f"Kenzie: Erro inesperado: {error}"
            )

            print()


# ============================================================
# INICIAR
# ============================================================

if __name__ == "__main__":

    main()